# 5.15 · 在线学习 + 概率校准（收官）/ Online Learning + Calibration (Part 5 Finale)

> **课程定位 / Where this fits**
> Part 5 收官课, 两个**生产环境**主题。(1) **在线学习**: 数据流式到达、或大到放不进内存时, 用 `partial_fit`/SGD 增量训练。(2) **概率校准**: 很多模型(SVM、树、boosting)输出的"概率"并不可信——0.9 不代表 90% 真会发生。需求高(风控阈值、期望损失 2.10)时必须校准。
> Two production topics to close Part 5: online/incremental learning for streaming or out-of-memory data, and probability calibration so that a predicted 0.9 really means ~90%.

> 💡 **面试相关 / Interview-relevant**
> - "在线学习 / partial_fit 适用场景" ★★★★
> - "SGD 分类器与逻辑回归/SVM 的关系" ★★★★
> - "什么是概率校准 / 可靠性曲线" ★★★★★
> - "Platt scaling vs Isotonic 校准区别" ★★★★
> - "为什么 SVM/随机森林概率不准" ★★★★

---

## 学习目标 / Learning Objectives
1. 在线/增量学习: `partial_fit`、`SGDClassifier`、`classes_` 预声明。
2. SGD 通过 `loss` 统一逻辑回归/SVM/感知机。
3. **概率校准**: 可靠性曲线、Brier 分数。
4. **Platt(sigmoid) vs Isotonic** 校准(接 4.16 等张回归)。
5. Part 5 全景回顾。

## 目录 / TOC
1. [在线学习与 partial_fit ⭐](#1)
2. [🩺 数据 + 流式训练](#2)
3. [SGD 统一多种线性模型](#3)
4. [概率校准: 可靠性曲线 ⭐](#4)
5. [Platt vs Isotonic ⭐](#5)
6. [Part 5 全景回顾](#6)


<a id="1"></a>
## 1. 在线学习与 partial_fit ⭐ / Online Learning

**批量学习**(前 14 课): 一次性把全部数据喂进 `fit`。**在线/增量学习**: 数据**一块一块**来, 模型用 `partial_fit` 持续更新, 永远不需要一次装下全部数据。适用:
- **数据流**: 实时日志、点击流, 边来边学。
- **超大数据**: 放不进内存, 分块(mini-batch)读入。
- **概念漂移**: 数据分布随时间变, 模型需持续适应。

sklearn 支持 `partial_fit` 的模型: `SGDClassifier`、`PassiveAggressive`、`Perceptron`、`MultinomialNB`、`MiniBatchKMeans` 等。关键: **首次** `partial_fit` 要传 `classes=` 预声明所有类别(因为它看不到全量数据)。


<a id="2"></a>
## 2. 数据 + 流式训练 / Streaming Simulation

回到 **Breast Cancer**(5.1 介绍过), 把训练集切成小批模拟数据流, 用 `SGDClassifier` 增量学。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
sns.set_theme(style="whitegrid")

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3,
                                          stratify=data.target, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)

# 流式: 把训练集切成 20 个 mini-batch / stream in mini-batches
sgd = SGDClassifier(loss="log_loss", random_state=0)  # log_loss → 在线逻辑回归
batches = np.array_split(np.arange(len(Xtr)), 20)
acc_curve = []
for i, b in enumerate(batches):
    sgd.partial_fit(Xtr[b], y_tr[b], classes=[0, 1])   # 首批必须给 classes
    acc_curve.append(sgd.score(Xte, y_te))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 21), acc_curve, "o-")
ax.set_xlabel("已学习的 mini-batch 数"); ax.set_ylabel("test 准确率")
ax.set_title("在线学习: 每来一批 partial_fit 更新, 准确率逐步爬升")
plt.tight_layout(); plt.show()
print(f"流式学完 20 批后 test 准确率: {acc_curve[-1]:.3f} (从未一次装入全部数据)")


<a id="3"></a>
## 3. SGD 统一多种线性模型 / SGD Unifies Linear Models

`SGDClassifier` 用随机梯度下降优化, 换 `loss` 就变成不同模型——前面学的线性分类器其实是"同一台引擎配不同损失":
- `loss="log_loss"` → **逻辑回归**(5.1)
- `loss="hinge"` → **线性 SVM**(5.5)
- `loss="perceptron"` → **感知机**
都能 `partial_fit` 在线训练, 大数据上比闭式/批量求解更省内存。


In [ ]:
for loss in ["log_loss", "hinge", "perceptron"]:
    m = SGDClassifier(loss=loss, random_state=0, max_iter=1000).fit(Xtr, y_tr)
    name = {"log_loss":"逻辑回归","hinge":"线性SVM","perceptron":"感知机"}[loss]
    print(f"SGD(loss={loss:<11}) = {name:<8} test 准确率 {m.score(Xte, y_te):.3f}")
print("\n同一个 SGD 引擎, 换损失=换模型; 全部支持 partial_fit 在线学习")


<a id="4"></a>
## 4. 概率校准: 可靠性曲线 ⭐ / Calibration & Reliability Curve

模型说"0.9", 这 0.9 可信吗? **校准良好**意味着: 在所有预测 ≈0.9 的样本里, 真有 ≈90% 是正类。

- **逻辑回归**: 用对数损失(=正确概率目标)训练, 天然**校准较好**。
- **SVM**: 输出的是到边界的距离, 不是概率, **校准差**。
- **随机森林/boosting**: 倾向把概率推向极端(0/1), 中段不准。
- **朴素贝叶斯**: 独立假设错→概率常过度自信。

**可靠性曲线(reliability/calibration curve)**: x=预测概率分箱, y=该箱实际正类比例。对角线=完美校准。**Brier 分数**(预测概率与 0/1 的均方误差)量化校准, 越低越好。


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "逻辑回归": LogisticRegression(max_iter=2000),
    "SVM(rbf)": SVC(probability=True, random_state=0),
    "随机森林": RandomForestClassifier(n_estimators=200, random_state=0),
}
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot([0,1],[0,1],"k--", label="完美校准")
for name, m in models.items():
    m.fit(Xtr, y_tr); p = m.predict_proba(Xte)[:,1]
    frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (Brier={brier_score_loss(y_te, p):.3f})")
ax.set_xlabel("预测概率"); ax.set_ylabel("实际正类比例"); ax.legend()
ax.set_title("可靠性曲线: 越贴对角线越校准好; 逻辑回归通常最贴")
plt.tight_layout(); plt.show()
print("逻辑回归 Brier 最低(校准最好); SVM/RF 偏离对角线 → 概率不可直接当真")


<a id="5"></a>
## 5. Platt vs Isotonic 校准 ⭐ / Calibrating Probabilities

`CalibratedClassifierCV` 在一个**留出的校准集**上, 学一个把"原始分数 → 校准概率"的映射:
- **Platt scaling(sigmoid)**: 拟合一个 sigmoid(逻辑回归)。参数少(2个), **小数据稳**, 假设校准误差是 sigmoid 形。
- **Isotonic(等张)**: 拟合一个**单调非降**的分段函数(正是 4.16 的等张回归!)。更灵活、能纠任意单调失真, 但**需更多数据**, 否则过拟合。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

svc = SVC(probability=True, random_state=0).fit(Xtr, y_tr)
cal_sig = CalibratedClassifierCV(SVC(random_state=0), method="sigmoid", cv=5).fit(Xtr, y_tr)
cal_iso = CalibratedClassifierCV(SVC(random_state=0), method="isotonic", cv=5).fit(Xtr, y_tr)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot([0,1],[0,1],"k--", label="完美校准")
for name, m in [("SVM 未校准", svc), ("Platt(sigmoid)", cal_sig), ("Isotonic", cal_iso)]:
    p = m.predict_proba(Xte)[:,1]
    fp, mp = calibration_curve(y_te, p, n_bins=8, strategy="quantile")
    ax.plot(mp, fp, "o-", label=f"{name} (Brier={brier_score_loss(y_te, p):.3f})")
ax.set_xlabel("预测概率"); ax.set_ylabel("实际正类比例"); ax.legend()
ax.set_title("校准前后: Platt/Isotonic 把 SVM 分数拉回对角线")
plt.tight_layout(); plt.show()
print("校准后 Brier 下降、曲线更贴对角线; 需要可信概率(风控/期望损失2.10)时务必校准")
print("Platt: 少参数小数据稳; Isotonic: 灵活需更多数据(就是 4.16 的等张回归)")


<a id="6"></a>
## 6. Part 5 全景回顾 / Part 5 Big Picture

```
分类器全家 (判别式 / 生成式):
  线性: 逻辑回归(5.1) softmax(5.2) 线性SVM(5.5) — 边界=超平面
  生成: 朴素贝叶斯(5.4) LDA/QDA(5.12) — 建模 P(x|c)
  惰性: KNN(5.3) — 不训练, 近邻投票
  核: 核SVM(5.5) — 隐式升维画非线性边界
  树与集成: 决策树(5.6) → 随机森林(5.7, bagging降方差)
            → GBDT(5.8) → XGBoost(5.9)/LightGBM(5.10)/CatBoost(5.11) (boosting降偏差)
评估: 混淆矩阵 P/R/F1 ROC-AUC PR-AUC(5.1); micro/macro(5.13)
实战: 多类多标签(5.13) 不平衡(5.14) 在线学习+校准(5.15)
贯穿主线: 损失=负对数似然; 正则=复杂度控制; 偏差方差权衡; 防泄漏只 fit 训练折
```

**选型直觉**:
- 要可解释/线性基线 → 逻辑回归
- 表格数据要精度 → GBDT 三巨头(默认 LightGBM/XGBoost)
- 文本/高维稀疏 → 线性SVM / 朴素贝叶斯
- 小数据光滑边界 → 核SVM / 判别分析
- 不平衡 → class_weight + PR-AUC + 阈值; 要可信概率 → 校准

### 💡 面试速查
1. **在线学习**(partial_fit): 流式/超大/概念漂移; 首批传 classes
2. **SGDClassifier** 换 loss = 逻辑回归/SVM/感知机
3. **校准**: 预测 0.9 真有 90%? 看可靠性曲线 + Brier; LR 天然好, SVM/RF 差
4. **Platt(sigmoid, 小数据) vs Isotonic(等张, 更灵活需更多数据)**

### Part 5 完成 🎉
监督学习分类全部走通。下一部分 **Part 6 无监督学习**(聚类、降维、异常检测), 进入"没有标签"的世界。
